# 从 PokemonDB 抓取前 151 只宝可梦数据

本 notebook 会从 https://pokemondb.net 抓取国家图鉴 1~151 号宝可梦的基础信息、属性、种族值、中文名和进化链信息，并导出为 `../data/pokemon151.csv`。

In [8]:
import json
import re
import time
from pathlib import Path

try:
    import requests
    from bs4 import BeautifulSoup
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4'])
    import requests
    from bs4 import BeautifulSoup

BASE_URL = 'https://pokemondb.net'
NATIONAL_URL = f'{BASE_URL}/pokedex/national'
OUTPUT_JSON = Path('../data/pokemon151.json').resolve()
OUTPUT_CSV = Path('../data/pokemon151.csv').resolve()
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

session = requests.Session()
session.headers.update(HEADERS)

def get_soup(url: str):
    response = session.get(url, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

def parse_national_list():
    soup = get_soup(NATIONAL_URL)
    cards = soup.select('div.infocard')
    pokemon_list = []
    for card in cards:
        small = card.select_one('.infocard-lg-data small')
        if small is None:
            continue
        match = re.search(r'#(\d+)', small.get_text())
        if not match:
            continue
        national_id = int(match.group(1))
        if national_id > 151:
            continue
        name_el = card.select_one('.ent-name')
        if name_el is None:
            continue
        name = name_el.get_text(strip=True)
        href_el = card.select_one('a[href^="/pokedex/"]')
        if href_el is None:
            continue
        slug = href_el['href'].strip('/').split('/')[-1]
        detail_url = BASE_URL + href_el['href']
        types = [type_el.get_text(strip=True) for type_el in card.select('.itype')]
        pokemon_list.append({
            'pokemon_id': national_id,
            'name_en': name,
            'slug': slug,
            'detail_url': detail_url,
            'type1': types[0] if types else '',
            'type2': types[1] if len(types) > 1 else '',
        })
    return sorted(pokemon_list, key=lambda item: item['pokemon_id'])

def parse_vitals(soup):
    result = {}
    table = soup.select_one('table.vitals-table')
    if not table:
        return result
    for row in table.select('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        key = th.get_text(strip=True)
        value = ' '.join(td.stripped_strings)
        result[key] = value
    return result

def parse_chinese_name(soup):
    heading = soup.find(lambda tag: tag.name in ['h2', 'h3'] and 'Other languages' in tag.get_text())
    if heading is None:
        return ''
    table = heading.find_next('table')
    if table is None:
        return ''
    for row in table.select('tr'):
        th = row.find('th')
        td = row.find('td')
        if not th or not td:
            continue
        if th.get_text(strip=True) == 'Chinese (Simplified)':
            return td.get_text(strip=True)
    return ''

def parse_base_stats(soup):
    stats = {}
    heading = soup.find(lambda tag: tag.name in ['h2', 'h3'] and 'Base stats' in tag.get_text())
    if heading is None:
        return stats
    table = heading.find_next('table')
    if table is None:
        return stats
    for row in table.select('tr'):
        cells = [cell.get_text(strip=True) for cell in row.find_all(['th', 'td'])]
        if len(cells) < 2:
            continue
        stat_name = cells[0].replace('.', '').replace(' ', '_').lower()
        stat_value = cells[1] if cells[1].isdigit() else cells[1].split()[0] if cells[1] else ''
        if stat_name and stat_value:
            stats[stat_name] = stat_value
    return stats

def parse_latest_generation(local_numbers: str):
    version_map = {
        'scarlet': 9,
        'violet': 9,
        'legends: z-a': 9,
        'sword': 8,
        'shield': 8,
        'brilliant diamond shining pearl': 8,
        'lets go pikachu': 8,
        'lets go eevee': 8,
        'legends: arceus': 8,
        'ultra sun': 7,
        'ultra moon': 7,
        'sun': 7,
        'moon': 7,
        'x': 6,
        'y': 6,
        'omega ruby alpha sapphire': 6,
        'black 2 white 2': 5,
        'black white': 5,
        'heartgold': 4,
        'soul silver': 4,
        'diamond pearl platinum': 4,
        'ruby sapphire': 3,
        'firered leafgreen': 3,
        'gold silver crystal': 2,
        'red blue': 1,
        'yellow': 1,
    }
    text = local_numbers.lower()
    generation = 0
    for key, value in version_map.items():
        if key in text:
            generation = max(generation, value)
    return str(generation) if generation else ''

def parse_evolution_chain(slug_to_id):
    api_session = requests.Session()
    api_session.headers.update({'User-Agent': HEADERS['User-Agent'], 'Accept': 'application/json'})
    chain_cache = {}
    species_cache = {}
    chain_info_map = {}

    def fetch_json(url: str):
        response = api_session.get(url, timeout=20)
        response.raise_for_status()
        return response.json()

    for slug in slug_to_id:
        species_url = f'https://pokeapi.co/api/v2/pokemon-species/{slug}'
        try:
            species_cache[slug] = fetch_json(species_url)
        except requests.HTTPError:
            continue
        chain_url = species_cache[slug].get('evolution_chain', {}).get('url')
        if not chain_url:
            continue
        if chain_url not in chain_cache:
            chain_cache[chain_url] = fetch_json(chain_url)
        time.sleep(0.2)

    def traverse(node, stage, parent_slug):
        name = node['species']['name']
        post = [child['species']['name'] for child in node['evolves_to']]
        chain_nodes.append({
            'slug': name,
            'stage': stage,
            'pre': parent_slug,
            'post': post,
        })
        for child in node['evolves_to']:
            traverse(child, stage + 1, name)

    for chain_url, chain_data in chain_cache.items():
        chain_nodes = []
        traverse(chain_data['chain'], 1, None)
        max_stage = max(node['stage'] for node in chain_nodes) if chain_nodes else 1
        chain_id = int(chain_url.rstrip('/').split('/')[-1])
        for node in chain_nodes:
            chain_info_map[node['slug']] = {
                'evolution_id': chain_id,
                'evolution_stage': node['stage'],
                'remaining_evolutions': max_stage - node['stage'],
                'pre_slug': node['pre'],
                'post_slugs': node['post'],
            }
    return chain_info_map

def parse_pokemon_detail(pokemon):
    detail_soup = get_soup(pokemon['detail_url'])
    vitals = parse_vitals(detail_soup)
    name_cn = parse_chinese_name(detail_soup)
    stats = parse_base_stats(detail_soup)
    local_numbers = vitals.get('Local №', '')
    latest_generation = parse_latest_generation(local_numbers)
    return {
        'pokemon_id': pokemon['pokemon_id'],
        'slug': pokemon['slug'],
        'name_en': pokemon['name_en'],
        'name_cn': name_cn,
        'type1': pokemon['type1'],
        'type2': pokemon['type2'],
        'hp': stats.get('hp', ''),
        'attack': stats.get('attack', ''),
        'defense': stats.get('defense', ''),
        'sp_attack': stats.get('sp', stats.get('sp_atk', stats.get('sp_attack', ''))),
        'sp_defense': stats.get('sp_def', stats.get('sp_defense', '')),
        'speed': stats.get('speed', ''),
        'local_numbers': local_numbers,
        'latest_generation': latest_generation,
    }

def fetch_all_pokemon():
    pokemon_list = parse_national_list()
    slug_to_id = {pokemon['slug']: pokemon['pokemon_id'] for pokemon in pokemon_list}
    chain_info_map = parse_evolution_chain(slug_to_id)
    results = []
    for index, pokemon in enumerate(pokemon_list, start=1):
        print(f"Fetching {pokemon['pokemon_id']}: {pokemon['name_en']} ({index}/{len(pokemon_list)})")
        detail = parse_pokemon_detail(pokemon)
        chain_info = chain_info_map.get(detail['slug'], {})
        pre_id = slug_to_id.get(chain_info.get('pre_slug'), '') if chain_info.get('pre_slug') else ''
        post_ids = [str(slug_to_id[slug]) for slug in chain_info.get('post_slugs', []) if slug in slug_to_id]
        detail.update({
            'evolution_id': chain_info.get('evolution_id', ''),
            'evolution_stage': chain_info.get('evolution_stage', ''),
            'remaining_evolutions': chain_info.get('remaining_evolutions', ''),
            'pre_evolution_id': pre_id,
            'post_evolution_id': ','.join(post_ids),
        })
        results.append(detail)
        time.sleep(0.5)
    return results

def save_csv(pokemon_data):
    import csv
    fieldnames = [
        'pokemon_id',
        'name_en',
        'name_cn',
        'type1',
        'type2',
        'hp',
        'attack',
        'defense',
        'sp_attack',
        'sp_defense',
        'speed',
        'evolution_id',
        'evolution_stage',
        'remaining_evolutions',
        'pre_evolution_id',
        'post_evolution_id',
        'latest_generation',
    ]
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with open(OUTPUT_CSV, 'w', encoding='utf-8', newline='') as fp:
        writer = csv.DictWriter(fp, fieldnames=fieldnames)
        writer.writeheader()
        for row in pokemon_data:
            writer.writerow({key: row.get(key, '') for key in fieldnames})

pokemon_data = fetch_all_pokemon()
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, 'w', encoding='utf-8') as fp:
    json.dump(pokemon_data, fp, ensure_ascii=False, indent=2)
save_csv(pokemon_data)
print(f'完成：已保存 {len(pokemon_data)} 条宝可梦数据到 {OUTPUT_JSON} 和 {OUTPUT_CSV}')

Fetching 1: Bulbasaur (1/151)
Fetching 2: Ivysaur (2/151)
Fetching 3: Venusaur (3/151)
Fetching 4: Charmander (4/151)
Fetching 5: Charmeleon (5/151)
Fetching 6: Charizard (6/151)
Fetching 7: Squirtle (7/151)
Fetching 8: Wartortle (8/151)
Fetching 9: Blastoise (9/151)
Fetching 10: Caterpie (10/151)
Fetching 11: Metapod (11/151)
Fetching 12: Butterfree (12/151)
Fetching 13: Weedle (13/151)
Fetching 14: Kakuna (14/151)
Fetching 15: Beedrill (15/151)
Fetching 16: Pidgey (16/151)
Fetching 17: Pidgeotto (17/151)
Fetching 18: Pidgeot (18/151)
Fetching 19: Rattata (19/151)
Fetching 20: Raticate (20/151)
Fetching 21: Spearow (21/151)
Fetching 22: Fearow (22/151)
Fetching 23: Ekans (23/151)
Fetching 24: Arbok (24/151)
Fetching 25: Pikachu (25/151)
Fetching 26: Raichu (26/151)
Fetching 27: Sandshrew (27/151)
Fetching 28: Sandslash (28/151)
Fetching 29: Nidoran♀ (29/151)
Fetching 30: Nidorina (30/151)
Fetching 31: Nidoqueen (31/151)
Fetching 32: Nidoran♂ (32/151)
Fetching 33: Nidorino (33/151)
Fet